# Profile Headers Analytics

This notebook helps you:

1. Run `ci/profile_headers.sh` from the repo root.
2. Load and inspect the generated CSV.
3. Explore expensive headers by compile time, transitive LOC, and include count.
4. Build combined ranking scores to identify high-impact headers.

Expected CSV columns:
- `header_path`
- `compile_time_ms`
- `transitive_loc`
- `included_by_header_count`

In [29]:
%pip install pandas matplotlib plotly nbformat


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
import os
import shlex
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Could not locate repo root (.git). Start notebook from inside the CCCL repo."
    )

print(f"Repo root: {REPO_ROOT}")
print(f"Python executable: {sys.executable}")

Repo root: /home/coder/cccl
Python executable: /home/coder/.local/share/venvs/cccl/bin/python


In [31]:
# --- Run profile_headers.sh ---
import subprocess
from pathlib import Path

if "REPO_ROOT" not in globals():
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
        REPO_ROOT = REPO_ROOT.parent

output_csv = Path("/tmp/profile_headers.csv")
cmd = [
    "bash",
    str(REPO_ROOT / "ci" / "profile_headers.sh"),
    "--output-csv",
    str(output_csv),
]

print("Command:")
print("  " + " ".join(shlex.quote(x) for x in cmd))

subprocess.run(cmd, cwd=REPO_ROOT, check=True)
print(f"\nWrote: {output_csv}")

Command:
  bash /home/coder/cccl/ci/profile_headers.sh --output-csv /tmp/profile_headers.csv
-- CMAKE_CXX_STANDARD: 17
-- CMAKE_CUDA_STANDARD: 17
-- Creating symlink from /home/coder/cccl/compile_commands.json to /home/coder/cccl/build/cuda13.1-gcc14/profile-headers/compile_commands.json...
-- Found Thrust: /home/coder/cccl/lib/cmake/thrust/thrust-config.cmake (found suitable version "3.4.0.0", minimum required is "3.4.0.0")
-- Found CUB: /home/coder/cccl/lib/cmake/cub/cub-config.cmake (found suitable version "3.4.0.0", minimum required is "3.4.0.0")


[profile-headers] Configuring and building preset 'profile-headers'...


-- LLVM host triple: x86_64-unknown-linux-gnu
-- LLVM default target triple: x86_64-unknown-linux-gnu
-- CPM: Adding package Catch2@3.12.0 (v3.12.0)
-- Finding CCCL components: Thrust;CUB;libcudacxx;cudax
-- Found libcudacxx: /home/coder/cccl/lib/cmake/libcudacxx/libcudacxx-config.cmake (found suitable exact version "3.4.0.0")
-- Found cudax: /home/coder/cccl/lib/cmake/cudax/cudax-config.cmake (found suitable exact version "3.4.0.0")
-- CPM: Adding package dlpack@1.2 (v1.2)
-- C++11 support has been enabled by default.
-- Lit enabled CUDA architectures: native
-- libcudacxx_LIT_FLAGS: 
-- Found CUB: /home/coder/cccl/lib/cmake/cub/cub-config.cmake (found version "3.4.0.0")
-- Found libcudacxx: /home/coder/cccl/lib/cmake/libcudacxx/libcudacxx-config.cmake (found version "3.4.0.0")
-- Found Thrust: /home/coder/cccl/lib/cmake/thrust/thrust-config.cmake (found version "3.4.0.0")


-- /usr/bin/FileCheck found... building atomic codegen tests


-- Enabling Thrust configuration: CPP.CUDA
-- 1 unique Thrust host.device configurations generated
-- CPP system found?  TRUE
-- CUDA system found? TRUE
-- TBB system found?  FALSE
-- OMP system found?  FALSE
-- Found cudax: /home/coder/cccl/lib/cmake/cudax/cudax-config.cmake (found version "3.4.0.0")
-- Configuring done (2.6s)
-- Generating done (1.1s)
-- Build files have been written to: /home/coder/cccl/build/cuda13.1-gcc14/profile-headers
[0/2] Re-checking globbed directories...
ninja: no work to do.


[profile-headers] Discovering preprocessed header TUs...
[profile-headers] Found 874 header TUs.
[profile-headers] Counting direct include references across CCCL headers...
[profile-headers] Counting transitive LOC with cloc...



Wrote: /tmp/profile_headers.csv


[profile-headers] Writing CSV rows...
[profile-headers] Done. Profile headers CSV: /tmp/profile_headers.csv


In [32]:
# --- Load CSV ---
csv_path = Path("/tmp/profile_headers.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Missing CSV: {csv_path}. Run the script first.")

df = pd.read_csv(csv_path)

# Defensive numeric conversion for robust downstream analysis.
for col in ["compile_time_ms", "transitive_loc", "included_by_header_count"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(5)

Rows: 874
Columns: ['header_path', 'compile_time_ms', 'transitive_loc', 'included_by_header_count']


,header_path,compile_time_ms,transitive_loc,included_by_header_count
0,cub/agent/agent_adjacent_difference.cuh,9500,127975,2
1,cub/agent/agent_batched_topk.cuh,9643,135216,2
2,cub/agent/agent_batch_memcpy.cuh,10041,136253,2
3,cub/agent/agent_find.cuh,10686,137117,1
4,cub/agent/agent_for.cuh,6294,73089,3


In [33]:
# --- Quick top-N views ---
TOP_N = 5

print("Top by compile_time_ms")
display(
    df.nlargest(TOP_N, "compile_time_ms")[
        ["header_path", "compile_time_ms", "transitive_loc", "included_by_header_count"]
    ]
)

print("\nTop by transitive_loc")
display(
    df.nlargest(TOP_N, "transitive_loc")[
        ["header_path", "transitive_loc", "compile_time_ms", "included_by_header_count"]
    ]
)

print("\nTop by included_by_header_count")
display(
    df.nlargest(TOP_N, "included_by_header_count")[
        ["header_path", "included_by_header_count", "compile_time_ms", "transitive_loc"]
    ]
)

Top by compile_time_ms


,header_path,compile_time_ms,transitive_loc,included_by_header_count
50,cub/cub.cuh,19223,254569,0
273,cub/cub.cuh,19126,254569,0
509,cuda/std/__pstl_algorithm,17967,230202,0
723,cuda/std/__pstl_algorithm,17906,230202,0
555,thrust/extrema.h,17342,235697,1



Top by transitive_loc


,header_path,transitive_loc,compile_time_ms,included_by_header_count
50,cub/cub.cuh,254569,19223,0
273,cub/cub.cuh,254569,19126,0
604,thrust/partition.h,251411,16092,0
818,thrust/partition.h,251411,14188,0
624,thrust/sort.h,249440,14012,6



Top by included_by_header_count


,header_path,included_by_header_count,compile_time_ms,transitive_loc
49,cub/config.cuh,221,5669,70364
272,cub/config.cuh,221,5727,70364
488,cuda/std/cstdint,214,1405,24044
702,cuda/std/cstdint,214,1424,24044
453,cuda/__cccl_config,125,1450,23850


In [34]:
# --- Interactive scatter (hover shows header name) ---

import plotly.express as px

plot_df = df.copy()

fig = px.scatter(
    plot_df,
    x="included_by_header_count",
    y="compile_time_ms",
    hover_name="header_path",
    hover_data={
        "transitive_loc": True,
        "included_by_header_count": True,
        "compile_time_ms": ":.2f",
        "header_path": False,
    },
    opacity=0.6,
    title="Include count vs compile time (hover for header)",
)

fig.update_layout(
    xaxis_title="included_by_header_count",
    yaxis_title="compile_time_ms",
    height=900,
)
fig.show()

In [35]:
# --- Optional: run ctadvisor and parse expensive headers ---
CTADVISOR_ENTRIES = 10
CTADVISOR_THREADS = os.cpu_count() or 8

trace_root = (
    REPO_ROOT
    / "build"
    / os.environ.get("CCCL_BUILD_INFIX", "cuda13.1-gcc14")
    / "profile-headers"
    / "header_testing"
    / "device_time_trace"
)
ctadvisor_cmd = [
    "ctadvisor",
    "--trace-file-path",
    str(trace_root),
    "--header-advisor-entries",
    str(CTADVISOR_ENTRIES),
    "--thread-number",
    str(CTADVISOR_THREADS),
]

print("Command:")
print("  " + " ".join(shlex.quote(x) for x in ctadvisor_cmd))

result = subprocess.run(
    ctadvisor_cmd, cwd=REPO_ROOT, check=True, capture_output=True, text=True
)
print(result.stdout)

Command:
  ctadvisor --trace-file-path /home/coder/cccl/build/cuda13.1-gcc14/profile-headers/header_testing/device_time_trace --header-advisor-entries 10 --thread-number 64
Trace file loading is complete
***********************************************
Start expensive template advisor
Top template functions/classes by instantiation time:
  [10.67s]: (1596 times in 1596 files) std::__cxx11::basic_string 
    [2.80s]: (399 times in 399 files) std::__cxx11::basic_string<char32_t, std::char_traits<char32_t>, std::allocator<char32_t>>
    [2.77s]: (399 times in 399 files) std::__cxx11::basic_string<char16_t, std::char_traits<char16_t>, std::allocator<char16_t>>
    [2.55s]: (399 times in 399 files) std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char>>
    [2.53s]: (399 times in 399 files) std::__cxx11::basic_string<int, std::char_traits<int>, std::allocator<int>>
  [7.86s]: (3622 times in 3622 files) std::__cxx11::basic_string::basic_string 
    [1.81s]: (399 times i